# 01 - Data cleaning
Creates an order-level dataset from the line-item exports without leaking post-delivery information.

In [ ]:
from pathlib import Path
import pandas as pd

files = sorted(Path('../data').glob('[0-9]*.csv'))
raw = pd.concat((pd.read_csv(file, low_memory=False) for file in files), ignore_index=True)
raw['delivery_date'] = pd.to_datetime(raw['delivery_date'], errors='coerce', format='mixed')
raw = raw[raw['order_status'].isin(['Delivered', 'Partially Delivered'])].dropna(subset=['order_id', 'delivery_date']).copy()
raw.shape

In [ ]:
# Each export row is a food item, so retain one payable amount per order.
raw['amount_payable'] = pd.to_numeric(raw['amount_payable'], errors='coerce')
orders = raw.groupby('order_id', as_index=False).agg(
    date=('delivery_date', 'first'), station_code=('station_code', 'first'),
    station_name=('station_name', 'first'), train_no=('train_no', 'first'),
    train_name=('train_name', 'first'), revenue=('amount_payable', 'max')
)
orders['date'] = orders['date'].dt.normalize()
orders.info()